# Movie Recommender via Neural Collaborative Filtering

## NeuMF training and evaluation notebook

This notebook implements the model-training portion of the project requirements using MovieLens 25M and PyTorch. It trains NeuMF, evaluates a lightweight popularity baseline, performs chronological evaluation, saves model artifacts, and provides recommendation and preference-score helpers.

NeuMF combines GMF and MLP branches inside one model. Standalone GMF and MLP training runs are omitted to reduce training time. The project Markdown is a requirements reference, not executable notebook instruction. Full-data training is opt-in; no trained weights or final metrics are claimed until it completes.

## Notebook map and completion boundary

Run from top to bottom. The notebook covers MovieLens loading and validation, implicit interactions, chronological Leave-One-Out splits, a popularity baseline, NeuMF training, validation-based early stopping, final test evaluation, inference helpers, and model artifacts.

NeuMF is the only trainable model. The popularity baseline is a quick reference and does not require training. FastAPI, React, deployment, CI/CD, and full-stack tests remain separate project deliverables. Full training is opt-in because MovieLens 25M can take a long time.

### Setup, reproducibility, and compute device
The setup finds the project root from either the repository or ML folder, defines preprocessing and training settings, and fixes random seeds. CUDA is preferred whenever available; the cell prints the actual GPU name or a clear CPU fallback. Full training is controlled by RUN_FULL_TRAINING.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import random

import numpy as np
import pandas as pd
import torch
from torch import nn

root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next((root.resolve() for root in root_candidates
    if (root / 'Dataset' / 'ml-25m' / 'ratings.csv').exists()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter in the project or ML folder.'
DATA_DIR = PROJECT_ROOT / 'Dataset' / 'ml-25m'
ARTIFACT_DIR = PROJECT_ROOT / 'backend' / 'model_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
POSITIVE_THRESHOLD = 3.5
MIN_USER_RATINGS = 5
MIN_USER_INTERACTIONS = MIN_USER_RATINGS
MIN_POSITIVE_INTERACTIONS = 3
NEGATIVES_PER_POSITIVE = 4
EVAL_NEGATIVES = 99
EMBEDDING_DIM = 32
MLP_HIDDEN_DIMS = (64, 32, 16, 8)
DROPOUT = 0.2
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 3
MIN_NDCG_IMPROVEMENT = 1e-5
EVAL_USER_BATCH_SIZE = 256
MODEL_VARIANTS = ('NeuMF',)
RUN_FULL_TRAINING = True
MODEL_VERSION = '1.0.0'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f'TRAINING DEVICE: GPU — {torch.cuda.get_device_name(device)}')
    print(f'CUDA runtime: {torch.version.cuda}')
else:
    print('TRAINING DEVICE: CPU — CUDA is unavailable to this PyTorch environment')
print(f'Project: {PROJECT_ROOT}')
print(f'Dataset: {DATA_DIR}')
print(f'Artifacts: {ARTIFACT_DIR}')

TRAINING DEVICE: GPU — NVIDIA GeForce RTX 4050 Laptop GPU
CUDA runtime: 12.8
Project: C:\Users\ACER\OneDrive\Desktop\movie Remommender using NCF
Dataset: C:\Users\ACER\OneDrive\Desktop\movie Remommender using NCF\Dataset\ml-25m
Artifacts: C:\Users\ACER\OneDrive\Desktop\movie Remommender using NCF\backend\model_artifacts


## 1. Load the core data

### Load ratings and movie metadata
Reads the two MovieLens tables. The ratings table is the interaction history; the movie table supplies titles and genres for readable EDA output. Prints basic dataset sizes.

In [2]:
ratings = pd.read_csv(
    DATA_DIR / 'ratings.csv',
    dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int64'},
)
movies = pd.read_csv(DATA_DIR / 'movies.csv')

print(f'Ratings: {len(ratings):,}')
print(f'Users: {ratings.userId.nunique():,}')
print(f'Movies with ratings: {ratings.movieId.nunique():,}')
print(f'Movies in metadata: {movies.movieId.nunique():,}')

Ratings: 25,000,095
Users: 162,541
Movies with ratings: 59,047
Movies in metadata: 62,423


### Validate required columns and rating values
Checks that both tables have the fields this notebook expects and that ratings fall on the MovieLens five-star scale. Reports missing values, duplicate user/movie pairs, and observed rating values.

In [3]:
required_ratings = {'userId', 'movieId', 'rating', 'timestamp'}
required_movies = {'movieId', 'title', 'genres'}

assert required_ratings.issubset(ratings.columns)
assert required_movies.issubset(movies.columns)
assert ratings['rating'].between(0.5, 5.0).all()

print('Missing rating values:', ratings.isna().sum().sum())
print('Duplicate user/movie pairs:', ratings.duplicated(['userId', 'movieId']).sum())
print('Rating values:', sorted(ratings['rating'].unique().tolist()))

Missing rating values: 0
Duplicate user/movie pairs: 0
Rating values: [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


### Summarize the data and inspect popular movies
Computes rating dates, central tendency, interaction-matrix density, and user activity. It also joins the ten most rated movie IDs to metadata so the results are interpretable.

In [4]:
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
user_counts = ratings.groupby('userId').size()
movie_counts = ratings.groupby('movieId').size()

global_summary = pd.Series({
    'mean_rating': ratings['rating'].mean(),
    'median_rating': ratings['rating'].median(),
    'start_date': ratings['datetime'].min().date(),
    'end_date': ratings['datetime'].max().date(),
    'matrix_density': len(ratings) / (ratings.userId.nunique() * ratings.movieId.nunique()),
    'users_with_5_or_more_ratings': (user_counts >= MIN_USER_INTERACTIONS).sum(),
})
print(global_summary.to_string())

popular_movies = (
    movie_counts.nlargest(10).rename('interaction_count').reset_index()
    .merge(movies[['movieId', 'title', 'genres']], on='movieId')
)
print('\nTop popular movies:')
print(popular_movies.to_string(index=False))

mean_rating                       3.533854
median_rating                          3.5
start_date                      1995-01-09
end_date                        2019-11-21
matrix_density                    0.002605
users_with_5_or_more_ratings        162541

Top popular movies:
 movieId  interaction_count                                     title                           genres
     356              81491                       Forrest Gump (1994)         Comedy|Drama|Romance|War
     318              81482          Shawshank Redemption, The (1994)                      Crime|Drama
     296              79672                       Pulp Fiction (1994)      Comedy|Crime|Drama|Thriller
     593              74127          Silence of the Lambs, The (1991)            Crime|Horror|Thriller
    2571              72674                        Matrix, The (1999)           Action|Sci-Fi|Thriller
     260              68717 Star Wars: Episode IV - A New Hope (1977)          Action|Adventure|Sci-Fi


## 2. Build implicit interactions and a temporal split

### Convert ratings to positive implicit interactions
Keeps ratings at or above the configured threshold, filters users with too few total ratings, sorts interactions, and assigns contiguous internal user/movie indices. The model learns these binary positive examples; negative examples are sampled later.

In [5]:
# Positive labels are ratings at or above the configured preference threshold.
eligible_users = set(user_counts[user_counts >= MIN_USER_RATINGS].index)
positive = ratings.loc[
    ratings.userId.isin(eligible_users) & (ratings.rating >= POSITIVE_THRESHOLD),
    ['userId', 'movieId', 'rating', 'timestamp'],
].copy()
positive = (positive.sort_values(['userId', 'timestamp', 'movieId'], kind='mergesort')
    .drop_duplicates(['userId', 'movieId'], keep='last'))
positive_counts = positive.groupby('userId').size()
eligible_positive_users = set(
    positive_counts[positive_counts >= MIN_POSITIVE_INTERACTIONS].index)
positive = positive[positive.userId.isin(eligible_positive_users)].copy()
positive = positive.sort_values(['userId', 'timestamp', 'movieId'], kind='mergesort')
positive['user_idx'] = positive.userId.astype('category').cat.codes.astype('int32')
positive['movie_idx'] = positive.movieId.astype('category').cat.codes.astype('int32')
print(f'Positive interactions: {len(positive):,}')
print(f'Eligible users: {positive.user_idx.nunique():,}')
print(f'Eligible movies: {positive.movie_idx.nunique():,}')

Positive interactions: 15,629,703
Eligible users: 162,148
Eligible movies: 47,396


### Make a per-user chronological train/validation/test split
For each retained user, the latest positive interaction is held out for test and the second latest for validation. Earlier positives go to training. Ties in timestamp use the current row ordering, so timestamp ties do not establish a strict order.

In [6]:
# Latest positive is test; the preceding positive is validation.
positive['reverse_rank'] = positive.groupby('user_idx', sort=False).cumcount(ascending=False) + 1
train_df = positive[positive.reverse_rank > 2].copy().reset_index(drop=True)
val_df = positive[positive.reverse_rank == 2].copy().reset_index(drop=True)
test_df = positive[positive.reverse_rank == 1].copy().reset_index(drop=True)
assert train_df.user_idx.nunique() == val_df.user_idx.nunique() == test_df.user_idx.nunique()
print(f'Train positives: {len(train_df):,}')
print(f'Validation targets/users: {len(val_df):,} / {val_df.user_idx.nunique():,}')
print(f'Test targets/users: {len(test_df):,} / {test_df.user_idx.nunique():,}')

Train positives: 15,305,407
Validation targets/users: 162,148 / 162,148
Test targets/users: 162,148 / 162,148


### Save raw-ID/internal-index mappings
Builds reversible user and movie ID maps, the model vocabulary sizes, and each user?s set of known positive items. Writes mapping JSON files for inference and for interpreting model indices.

In [7]:
# Save reversible mappings for inference and build history strictly by split.
user_values = positive[['userId', 'user_idx']].drop_duplicates().sort_values('user_idx')
movie_values = positive[['movieId', 'movie_idx']].drop_duplicates().sort_values('movie_idx')
user2idx = dict(zip(user_values.userId.astype(int), user_values.user_idx.astype(int)))
idx2user = {idx: raw for raw, idx in user2idx.items()}
movie2idx = dict(zip(movie_values.movieId.astype(int), movie_values.movie_idx.astype(int)))
idx2movie = {idx: raw for raw, idx in movie2idx.items()}
num_users, num_movies = len(user2idx), len(movie2idx)
train_seen = train_df.groupby('user_idx').movie_idx.agg(set).to_dict()
val_history = {u: set(items) for u, items in train_seen.items()}
test_history = {u: set(items) for u, items in train_seen.items()}
for row in val_df.itertuples(index=False):
    test_history[int(row.user_idx)].add(int(row.movie_idx))

for filename, mapping in {
    'user2id.json': user2idx, 'id2user.json': idx2user,
    'movie2id.json': movie2idx, 'id2movie.json': idx2movie,
}.items():
    (ARTIFACT_DIR / filename).write_text(json.dumps(mapping), encoding='utf-8')
(ARTIFACT_DIR / 'user_mapping.json').write_text(
    json.dumps({'raw_to_index': user2idx, 'index_to_raw': idx2user}), encoding='utf-8')
(ARTIFACT_DIR / 'movie_mapping.json').write_text(
    json.dumps({'raw_to_index': movie2idx, 'index_to_raw': idx2movie}), encoding='utf-8')
train_interactions = train_df[['user_idx', 'movie_idx']].to_numpy(dtype=np.int64)
print(f'Embedding vocabularies: {num_users:,} users × {num_movies:,} movies')
print(f'ID maps saved in {ARTIFACT_DIR}')

Embedding vocabularies: 162,148 users × 47,396 movies
ID maps saved in C:\Users\ACER\OneDrive\Desktop\movie Remommender using NCF\backend\model_artifacts


## 3. NeuMF model and negative sampling

This notebook trains one learned model: NeuMF, which combines Generalized Matrix Factorization and an MLP branch. Standalone GMF and MLP experiments are skipped. Training uses binary implicit labels, a configurable 1:4 negative ratio, and newly sampled negatives each epoch. Held-out validation/test interactions are not used to fit model weights.

### Define the NeuMF architecture

The GMF branch uses a 32-dimensional element-wise user/movie product. The MLP branch concatenates two 32-dimensional embeddings and applies 64 -> 32 -> 16 -> 8 layers with ReLU and dropout. NeuMF joins both branches and returns a logit for BCEWithLogitsLoss.

In [8]:
class NeuMF(nn.Module):
    def __init__(self, n_users, n_movies):
        super().__init__()
        self.gmf_user = nn.Embedding(n_users, EMBEDDING_DIM)
        self.gmf_movie = nn.Embedding(n_movies, EMBEDDING_DIM)
        self.mlp_user = nn.Embedding(n_users, EMBEDDING_DIM)
        self.mlp_movie = nn.Embedding(n_movies, EMBEDDING_DIM)
        layers, width = [], EMBEDDING_DIM * 2
        for hidden in MLP_HIDDEN_DIMS:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(DROPOUT)])
            width = hidden
        self.mlp = nn.Sequential(*layers)
        self.output = nn.Linear(EMBEDDING_DIM + MLP_HIDDEN_DIMS[-1], 1)
        for layer in self.modules():
            if isinstance(layer, nn.Embedding):
                nn.init.normal_(layer.weight, mean=0.0, std=0.01)
            elif isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, user_idx, movie_idx):
        gmf = self.gmf_user(user_idx) * self.gmf_movie(movie_idx)
        pair = torch.cat([self.mlp_user(user_idx), self.mlp_movie(movie_idx)], dim=1)
        mlp = self.mlp(pair)
        return self.output(torch.cat([gmf, mlp], dim=1)).squeeze(1)

print('Only trainable model: NeuMF')

Only trainable model: NeuMF


### Stream shuffled training batches with fresh negatives

The generator avoids storing five copies of the large training table. It shuffles positives and resamples negatives from items not positive in that user’s training history on every epoch. Held-out interactions are not consulted by the training sampler.

In [9]:
def iter_training_batches(interactions, seen_by_user, n_movies, epoch):
    rng = np.random.default_rng(SEED + int(epoch))
    order = rng.permutation(len(interactions))
    positives_per_batch = max(1, BATCH_SIZE // (1 + NEGATIVES_PER_POSITIVE))
    for start in range(0, len(order), positives_per_batch):
        group = interactions[order[start:start + positives_per_batch]]
        users, items, labels = [], [], []
        for user_idx, positive_idx in group:
            user_idx, positive_idx = int(user_idx), int(positive_idx)
            users.append(user_idx); items.append(positive_idx); labels.append(1.0)
            seen = seen_by_user[user_idx]
            if len(seen) >= n_movies:
                raise ValueError(f'No unseen training item for user index {user_idx}.')
            sampled = set()
            while len(sampled) < NEGATIVES_PER_POSITIVE:
                candidate = int(rng.integers(n_movies))
                if candidate not in seen and candidate != positive_idx:
                    sampled.add(candidate)
            for candidate in sampled:
                users.append(user_idx); items.append(candidate); labels.append(0.0)
        yield (np.asarray(users, dtype=np.int64),
               np.asarray(items, dtype=np.int64),
               np.asarray(labels, dtype=np.float32))

print(f'Negative ratio: 1:{NEGATIVES_PER_POSITIVE}; batch size: {BATCH_SIZE}')

Negative ratio: 1:4; batch size: 512


## 4. Fixed evaluation candidates and ranking metrics

For each user, evaluation ranks one held-out positive against 99 sampled negatives. Candidate arrays are created once with fixed seeds, then reused for NeuMF and all validation epochs. Test metrics are computed only after checkpoint selection.

### Build validation and test candidate lists

Validation negatives exclude training history. Test negatives exclude training plus the chronologically earlier validation interaction. Candidate zero is always the held-out positive. This keeps the evaluation history consistent with the point in time being evaluated.

In [10]:
def build_eval_candidates(frame, history, n_movies, negatives=EVAL_NEGATIVES, seed=SEED):
    pairs = (frame[['user_idx', 'movie_idx']].drop_duplicates('user_idx')
             .sort_values('user_idx').to_numpy(dtype=np.int64))
    result = np.empty((len(pairs), negatives + 1), dtype=np.int32)
    rng = np.random.default_rng(seed)
    for row_idx, (user_idx, target) in enumerate(pairs):
        user_idx, target = int(user_idx), int(target)
        result[row_idx, 0] = target
        forbidden = history[user_idx] | {target}
        if n_movies - len(forbidden) < negatives:
            raise ValueError('Not enough eligible items for the requested evaluation negatives.')
        sampled = set()
        while len(sampled) < negatives:
            item = int(rng.integers(n_movies))
            if item not in forbidden:
                sampled.add(item)
        result[row_idx, 1:] = np.fromiter(sampled, dtype=np.int32, count=negatives)
    return pairs[:, 0], result

val_users, val_candidates = build_eval_candidates(
    val_df, val_history, num_movies, seed=SEED + 100)
test_users, test_candidates = build_eval_candidates(
    test_df, test_history, num_movies, seed=SEED + 200)
print(f'Validation users/candidates: {len(val_users):,} × {val_candidates.shape[1]}')
print(f'Test users/candidates: {len(test_users):,} × {test_candidates.shape[1]}')

Validation users/candidates: 162,148 × 100
Test users/candidates: 162,148 × 100


### Evaluate HR@10 and NDCG@10

Scores candidates in user batches. Average-rank tie handling avoids giving the positive an automatic advantage when the popularity baseline assigns equal counts. The same metric functions are used for baseline and neural models.

In [11]:
def ranking_metrics(scores):
    target = scores[:, :1]
    other = scores[:, 1:]
    rank = 1 + (other > target).sum(axis=1) + 0.5 * (other == target).sum(axis=1)
    return {
        'users': int(len(rank)),
        'HR@10': float(np.mean(rank <= 10)),
        'NDCG@10': float(np.mean(np.where(rank <= 10, 1 / np.log2(rank + 1), 0.0))),
    }

def evaluate_model(model, users, candidates):
    model.eval()
    result = []
    with torch.inference_mode():
        for start in range(0, len(users), EVAL_USER_BATCH_SIZE):
            user_part = users[start:start + EVAL_USER_BATCH_SIZE]
            item_part = candidates[start:start + EVAL_USER_BATCH_SIZE]
            user_tensor = torch.as_tensor(
                np.repeat(user_part[:, None], item_part.shape[1], axis=1).reshape(-1),
                dtype=torch.long, device=device)
            item_tensor = torch.as_tensor(item_part.reshape(-1), dtype=torch.long, device=device)
            result.append(model(user_tensor, item_tensor)
                          .reshape(len(user_part), -1).cpu().numpy())
    return ranking_metrics(np.concatenate(result, axis=0))

def evaluate_popularity(users, candidates, interactions):
    counts = np.bincount(interactions[:, 1], minlength=num_movies)
    return ranking_metrics(counts[candidates])

popularity_val = evaluate_popularity(val_users, val_candidates, train_interactions)
popularity_test = evaluate_popularity(test_users, test_candidates, train_interactions)
print('Popularity validation:', popularity_val)
print('Popularity test:', popularity_test)

Popularity validation: {'users': 162148, 'HR@10': 0.9268446110960358, 'NDCG@10': 0.6664291301617926}
Popularity test: {'users': 162148, 'HR@10': 0.915688136763944, 'NDCG@10': 0.6509715126319294}


## 5. Train with AdamW and validation NDCG early stopping

The optimizer is AdamW with the project configuration. Each model trains for at most 20 epochs. The best checkpoint is selected using validation NDCG@10, with patience-based early stopping. The held-out test set is not used in this function.

### Define a full training run for one architecture

Returns the restored best model, per-epoch history, and selected epoch. The best state dict is written to a variant-specific checkpoint as soon as validation improves.

In [12]:
MODEL_CONFIG = {
    'model_version': MODEL_VERSION,
    'gmf_embedding_dimension': EMBEDDING_DIM,
    'mlp_embedding_dimension': EMBEDDING_DIM,
    'mlp_hidden_dimensions': list(MLP_HIDDEN_DIMS),
    'dropout': DROPOUT,
    'loss': 'BCEWithLogitsLoss',
    'optimizer': 'AdamW',
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'batch_size': BATCH_SIZE,
    'negative_samples_per_positive': NEGATIVES_PER_POSITIVE,
    'evaluation_negatives': EVAL_NEGATIVES,
    'maximum_epochs': MAX_EPOCHS,
    'early_stopping_metric': 'validation_NDCG@10',
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'device': str(device),
}

def train_neumf():
    model = NeuMF(num_users, num_movies).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    best_score, best_epoch, best_state = -float('inf'), 0, None
    stale_epochs, history = 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        loss_sum = torch.zeros((), device=device)
        count = 0
        for batch_users, batch_items, batch_labels in iter_training_batches(
                train_interactions, train_seen, num_movies, epoch):
            u = torch.as_tensor(batch_users, dtype=torch.long, device=device)
            i = torch.as_tensor(batch_items, dtype=torch.long, device=device)
            y = torch.as_tensor(batch_labels, dtype=torch.float32, device=device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(u, i), y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.detach() * len(batch_labels)
            count += len(batch_labels)
        train_loss = float((loss_sum / max(count, 1)).item())
        validation = evaluate_model(model, val_users, val_candidates)
        history.append({
            'epoch': epoch, 'train_loss': train_loss,
            'validation_HR@10': validation['HR@10'],
            'validation_NDCG@10': validation['NDCG@10'],
        })
        print(f'NeuMF epoch {epoch}/{MAX_EPOCHS}: loss={train_loss:.5f}, '
              f'val HR@10={validation["HR@10"]:.5f}, '
              f'val NDCG@10={validation["NDCG@10"]:.5f}')
        if validation['NDCG@10'] > best_score + MIN_NDCG_IMPROVEMENT:
            best_score, best_epoch = validation['NDCG@10'], epoch
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            torch.save({
                'model_state_dict': best_state,
                'model_variant': 'NeuMF',
                'best_epoch': best_epoch,
                'validation_metrics': validation,
                'model_config': MODEL_CONFIG,
            }, ARTIFACT_DIR / 'neumf_best.pth')
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= EARLY_STOPPING_PATIENCE:
                print(f'NeuMF: early stopping at epoch {epoch}.')
                break
    if best_state is None:
        raise RuntimeError('NeuMF produced no validation checkpoint.')
    model.load_state_dict(best_state)
    return model.eval(), history, best_epoch

### Train NeuMF only

Set RUN_FULL_TRAINING to True in the setup cell, then run this cell. NeuMF trains with the configured negative sampler, and validation NDCG@10 selects the checkpoint. The popularity baseline remains a quick reference; standalone GMF and MLP training are not run.

In [13]:
trained_models, training_histories, best_epochs, learned_metrics = {}, {}, {}, {}
if RUN_FULL_TRAINING:
    model, history, best_epoch = train_neumf()
    trained_models['NeuMF'] = model
    training_histories['NeuMF'] = history
    best_epochs['NeuMF'] = best_epoch
    learned_metrics['NeuMF'] = evaluate_model(model, test_users, test_candidates)
    print(f'NeuMF selected epoch: {best_epoch}; test: {learned_metrics["NeuMF"]}')
else:
    print('Training is OFF. Set RUN_FULL_TRAINING = True and rerun this cell to train NeuMF.')

comparison_rows = [{'model': 'Popularity', **popularity_test}]
comparison_rows.extend({'model': name, **values}
                       for name, values in learned_metrics.items())
comparison = pd.DataFrame(comparison_rows).sort_values('NDCG@10', ascending=False)
print(comparison.to_string(index=False))

NeuMF epoch 1/20: loss=0.13058, val HR@10=0.97016, val NDCG@10=0.77154
NeuMF epoch 2/20: loss=0.10806, val HR@10=0.97246, val NDCG@10=0.78586
NeuMF epoch 3/20: loss=0.10370, val HR@10=0.97397, val NDCG@10=0.79186
NeuMF epoch 4/20: loss=0.10158, val HR@10=0.97438, val NDCG@10=0.79514
NeuMF epoch 5/20: loss=0.10009, val HR@10=0.97441, val NDCG@10=0.79684
NeuMF epoch 6/20: loss=0.09884, val HR@10=0.97557, val NDCG@10=0.79934
NeuMF epoch 7/20: loss=0.09795, val HR@10=0.97521, val NDCG@10=0.80027
NeuMF epoch 8/20: loss=0.09728, val HR@10=0.97531, val NDCG@10=0.80187
NeuMF epoch 9/20: loss=0.09671, val HR@10=0.97511, val NDCG@10=0.80140
NeuMF epoch 10/20: loss=0.09635, val HR@10=0.97568, val NDCG@10=0.80482
NeuMF epoch 11/20: loss=0.09611, val HR@10=0.97604, val NDCG@10=0.80528
NeuMF epoch 12/20: loss=0.09580, val HR@10=0.97615, val NDCG@10=0.80651
NeuMF epoch 13/20: loss=0.09544, val HR@10=0.97485, val NDCG@10=0.80461
NeuMF epoch 14/20: loss=0.09550, val HR@10=0.97549, val NDCG@10=0.80623
N

## 6. Save reproducible model artifacts

The artifact folder receives the ID mappings, movie metadata, preprocessing/training config, actual measured metrics, model metadata, and model card. Neural weights and final values exist only when the full training cell has completed. Movie metadata uses CSV so no additional Parquet dependency is needed.

### Write configuration, metrics, metadata, and model card

All counts and metrics are generated from the current dataset/run. When training is skipped, the artifact explicitly marks model results as not run instead of filling in fabricated values.

In [14]:
movie_metadata = movie_values.merge(
    movies[['movieId', 'title', 'genres']], on='movieId', how='left'
).sort_values('movie_idx')
movie_metadata.to_csv(ARTIFACT_DIR / 'movie_metadata.csv', index=False)

preprocessing_config = {
    'dataset': 'MovieLens 25M',
    'dataset_directory': str(DATA_DIR),
    'positive_rule': f'rating >= {POSITIVE_THRESHOLD}',
    'minimum_total_ratings': MIN_USER_RATINGS,
    'minimum_positive_interactions': MIN_POSITIVE_INTERACTIONS,
    'split': 'per-user chronological; latest=test, second-latest=validation, earlier=train',
    'training_negative_sampling': {
        'ratio': f'1:{NEGATIVES_PER_POSITIVE}',
        'refresh': 'new negatives and shuffled positives each epoch',
        'excluded_history': 'training positives only',
    },
    'evaluation': {
        'protocol': 'one held-out positive plus sampled negatives',
        'negative_candidates': EVAL_NEGATIVES,
        'validation_seed': SEED + 100,
        'test_seed': SEED + 200,
    },
    'counts': {
        'ratings': int(len(ratings)), 'users': int(num_users),
        'movies': int(num_movies), 'positives': int(len(positive)),
        'train_positives': int(len(train_df)),
        'validation_users': int(len(val_df)), 'test_users': int(len(test_df)),
    },
}
results = {
    'popularity_baseline': {'validation': popularity_val, 'test': popularity_test},
    'models': learned_metrics if RUN_FULL_TRAINING else {
        name: {'status': 'not_run', 'reason': 'Enable RUN_FULL_TRAINING and execute training.'}
        for name in MODEL_VARIANTS
    },
}
for name, value in {
    'config.json': {'preprocessing': preprocessing_config, 'training': MODEL_CONFIG},
    'metrics.json': results,
}.items():
    (ARTIFACT_DIR / name).write_text(json.dumps(value, indent=2), encoding='utf-8')
if RUN_FULL_TRAINING:
    (ARTIFACT_DIR / 'training_history.json').write_text(
        json.dumps(training_histories, indent=2), encoding='utf-8')

metadata = {
    'model_version': MODEL_VERSION,
    'model_family': 'Neural Collaborative Filtering',
    'variants': list(MODEL_VARIANTS),
    'training_completed': bool(RUN_FULL_TRAINING),
    'training_timestamp_utc': datetime.now(timezone.utc).isoformat()
        if RUN_FULL_TRAINING else None,
    'dataset_counts': preprocessing_config['counts'],
    'model_config': MODEL_CONFIG,
    'best_epochs': best_epochs,
    'test_metrics': results['models'],
    'artifact_files': [
        'user2id.json', 'id2user.json', 'movie2id.json', 'id2movie.json',
        'user_mapping.json', 'movie_mapping.json', 'movie_metadata.csv',
        'config.json', 'metrics.json', 'metadata.json', 'model_card.md',
    ] + ([f'{name.lower()}_best.pth' for name in MODEL_VARIANTS]
         + ['training_history.json'] if RUN_FULL_TRAINING else []),
}
(ARTIFACT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
card = f'''# Model Card — MovieLens NCF

- Version: {MODEL_VERSION}
- Training completed: {bool(RUN_FULL_TRAINING)}
- Dataset: MovieLens 25M
- Positive rule: rating >= {POSITIVE_THRESHOLD}
- Evaluation: chronological Leave-One-Out, {EVAL_NEGATIVES} sampled negatives
- Metrics: HR@10 and NDCG@10
- Models: Popularity baseline and NeuMF

## Intended use
Educational movie ranking and recommendation. The output is an implicit preference score, not an explicit 1–5 rating or a guarantee of enjoyment.

## Limitations
Offline sampled-negative results do not guarantee online engagement. Collaborative filtering has cold-start limitations and can reproduce popularity bias. MovieLens may not represent a commercial streaming population.

See metrics.json for measured values. No estimated values are inserted by this notebook.
'''
(ARTIFACT_DIR / 'model_card.md').write_text(card, encoding='utf-8')
print(f'Artifacts saved under {ARTIFACT_DIR}')

Artifacts saved under C:\Users\ACER\OneDrive\Desktop\movie Remommender using NCF\backend\model_artifacts


## 7. Top-N recommendation and preference score demo

Once trained models exist in this kernel, these helpers return ranked movies for an eligible MovieLens user and score one user/movie pair. Scores are sigmoid-transformed implicit preference estimates, not star ratings. Previously observed movies are removed from Top-N recommendations.

### Define inference helpers

The helpers validate raw IDs, map them to embedding indices, score candidates in batches, and join movie titles/genres after ranking. The final cell demonstrates NeuMF when training was run.

In [15]:
def predict_preference(raw_user_id, raw_movie_id, model_name='NeuMF'):
    if model_name not in trained_models:
        raise ValueError(f'{model_name} is not trained in this notebook session.')
    if raw_user_id not in user2idx:
        raise KeyError(f'Unknown or ineligible user ID: {raw_user_id}')
    if raw_movie_id not in movie2idx:
        raise KeyError(f'Movie is outside the trained vocabulary: {raw_movie_id}')
    model = trained_models[model_name]
    model.eval()
    with torch.inference_mode():
        logits = model(
            torch.tensor([user2idx[raw_user_id]], dtype=torch.long, device=device),
            torch.tensor([movie2idx[raw_movie_id]], dtype=torch.long, device=device))
        score = torch.sigmoid(logits).item()
    return {'user_id': int(raw_user_id), 'movie_id': int(raw_movie_id),
            'prediction_type': 'preference_score', 'score': float(score),
            'model_name': model_name, 'model_version': MODEL_VERSION}

def recommend_top_n(raw_user_id, top_n=10, model_name='NeuMF', score_batch_size=8192):
    if model_name not in trained_models:
        raise ValueError(f'{model_name} is not trained in this notebook session.')
    if raw_user_id not in user2idx:
        raise KeyError(f'Unknown or ineligible user ID: {raw_user_id}')
    if not 1 <= top_n <= 100:
        raise ValueError('top_n must be between 1 and 100.')
    user_idx = user2idx[raw_user_id]
    seen = set(positive.loc[positive.user_idx == user_idx, 'movie_idx'].astype(int))
    items = np.asarray([i for i in range(num_movies) if i not in seen], dtype=np.int64)
    if not len(items):
        return pd.DataFrame(columns=['rank', 'movieId', 'title', 'genres', 'preference_score'])
    model = trained_models[model_name].eval()
    parts = []
    with torch.inference_mode():
        for start in range(0, len(items), score_batch_size):
            batch = items[start:start + score_batch_size]
            users = torch.full((len(batch),), user_idx, dtype=torch.long, device=device)
            item_tensor = torch.as_tensor(batch, dtype=torch.long, device=device)
            parts.append(torch.sigmoid(model(users, item_tensor)).cpu().numpy())
    scores = np.concatenate(parts)
    positions = np.argpartition(-scores, min(top_n, len(scores)) - 1)[:top_n]
    positions = positions[np.argsort(-scores[positions])]
    raw_movies = [idx2movie[int(items[pos])] for pos in positions]
    result = movies[movies.movieId.isin(raw_movies)][['movieId', 'title', 'genres']].copy()
    score_by_movie = {int(idx2movie[int(items[pos])]): float(scores[pos]) for pos in positions}
    result['preference_score'] = result.movieId.map(score_by_movie)
    result = result.sort_values('preference_score', ascending=False).reset_index(drop=True)
    result.insert(0, 'rank', np.arange(1, len(result) + 1))
    return result

if RUN_FULL_TRAINING and 'NeuMF' in trained_models:
    example_user = int(next(iter(user2idx)))
    print(recommend_top_n(example_user, 10).to_string(index=False))
else:
    print('Run full training to produce personalized recommendation results.')

 rank  movieId                                                     title                      genres  preference_score
    1      318                          Shawshank Redemption, The (1994)                 Crime|Drama          0.991874
    2      593                          Silence of the Lambs, The (1991)       Crime|Horror|Thriller          0.991126
    3     2959                                         Fight Club (1999) Action|Crime|Drama|Thriller          0.990866
    4     2858                                    American Beauty (1999)               Drama|Romance          0.990497
    5      527                                   Schindler's List (1993)                   Drama|War          0.990319
    6     4993 Lord of the Rings: The Fellowship of the Ring, The (2001)           Adventure|Fantasy          0.988315
    7      858                                     Godfather, The (1972)                 Crime|Drama          0.988268
    8     4226                                  

## Model-training completeness review against the project PRD

| Requirement | Notebook status |
| --- | --- |
| MovieLens ingestion, validation, and data statistics | Implemented |
| Positive implicit feedback and reversible ID maps | Implemented |
| Chronological Leave-One-Out train/validation/test | Implemented |
| Fresh configurable negative sampling | Implemented |
| Popularity baseline | Implemented |
| NeuMF training with GMF and MLP branches | Implemented |
| BCEWithLogitsLoss, AdamW, GPU-first execution | Implemented |
| Validation NDCG@10 checkpointing and early stopping | Implemented |
| HR@10 and NDCG@10 on fixed sampled candidates | Implemented |
| Model metadata, model card, metrics, catalog, mappings | Implemented; trained weights/metrics require the full run |
| Top-N and preference-score demonstrations | Implemented for trained in-memory models |

The NeuMF training code is complete and ready to run, but RUN_FULL_TRAINING defaults to False because full MovieLens 25M training can be lengthy. Turn it on and run the notebook to create the trained checkpoint and final metric values. Standalone GMF and MLP training runs are intentionally omitted. This completes the training scope; FastAPI, React, deployment, CI/CD, and end-to-end tests remain separate PRD work.